# Simulating the Amplitude-Based QLBM with a BGK Collision

The collisionless `ABQLBM` streams populations that are already stored as amplitudes.
`ABBGKQLBM` adds a $\tau=1$ BGK collision to every time step, so one circuit performs
a complete LBM cycle: collide, stream, and apply boundary conditions.

The collision is possible because the $D_2Q_9$ equilibrium becomes *exactly linear* in
six features once velocities are written as $u_x = U \sin\theta_x$ and
$u_y = U \sin\theta_y$. Those six features are prepared in superposition on five
qubits, and a single $32 \times 32$ unitary maps them onto the equilibrium populations.
Since a collision contracts the state, that unitary needs somewhere to put the lost
norm: an `ABBGKLattice` reserves one marker qubit, whose $|1\rangle$ sector holds the
physical populations and whose $|0\rangle$ sector absorbs the rest. Streaming and
reflection are then controlled on the marker so they only act on the physical sector.

This notebook simulates a decaying Taylor-Green vortex on a periodic lattice and
compares the result against a classical $D_2Q_9$ BGK solver.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit_aer import AerSimulator

from qlbm.components import (
    ABBGKQLBM,
    ABBGKInitialConditions,
    ABBGKMeasurement,
    EmptyPrimitive,
)
from qlbm.infra import QiskitRunner, SimulationConfig
from qlbm.lattice import ABBGKLattice
from qlbm.tools.utils import create_directory_and_parents

## The lattice

An `ABBGKLattice` is built from the same specification as any other amplitude-based
$D_2Q_9$ lattice. It reserves the marker qubit on its own, and carries the
`D2Q9AngleEncoding` that parameterizes the collision.

In [ ]:
NUM_GRIDPOINTS = 16

lattice = ABBGKLattice(
    {
        "lattice": {
            "dim": {"x": NUM_GRIDPOINTS, "y": NUM_GRIDPOINTS},
            "velocities": "d2q9",
        },
        "geometry": [],
    }
)

output_dir = f"qlbm-output/ab-bgk-taylor-green-d2q9-{NUM_GRIDPOINTS}x{NUM_GRIDPOINTS}"
create_directory_and_parents(output_dir)

encoding = lattice.encoding

print(lattice)
print(f"qubits                  = {lattice.circuit.num_qubits}")
print(f"velocity bound U        = {encoding.max_velocity}")
print(f"normalization alpha     = {encoding.alpha:.6f}")
print(f"branch amplitudes beta  = {np.round(encoding.beta, 4)}")

## Initial conditions

Unlike the collisionless initial conditions, which occupy a set of velocity channels,
`ABBGKInitialConditions` encodes a *macroscopic flow field*: every grid point receives
the branch-angle state that matches its own velocity, weighted by its own density.

One wavelength of a Taylor-Green vortex fits the periodic domain in each direction. The
amplitude has to stay within the velocity bound $U$ of the encoding.

In [ ]:
INITIAL_SPEED = 0.05

x = np.arange(NUM_GRIDPOINTS, dtype=float)[:, None]
y = np.arange(NUM_GRIDPOINTS, dtype=float)[None, :]
wavenumber = 2.0 * np.pi / NUM_GRIDPOINTS

density = np.ones((NUM_GRIDPOINTS, NUM_GRIDPOINTS))
velocity_x = INITIAL_SPEED * np.sin(wavenumber * x) * np.cos(wavenumber * y)
velocity_y = -INITIAL_SPEED * np.cos(wavenumber * x) * np.sin(wavenumber * y)

assert INITIAL_SPEED <= encoding.max_velocity

## The simulation configuration

The configuration follows the usual `qlbm` pattern. Two details are specific to this
algorithm:

* `ABBGKMeasurement` samples the grid, velocity, and marker registers *jointly*, because
  the macroscopic moments are weighted sums over velocity channels rather than a grid
  marginal.
* The state at the end of a time step holds populations, while the next step needs a
  branch-angle state, so the two cannot be chained unitarily. `ABBGKReinitializer`
  closes that loop classically, and the runner therefore requires
  `statevector_snapshots=True`.

In [ ]:
cfg = SimulationConfig(
    initial_conditions=ABBGKInitialConditions(lattice, density, velocity_x, velocity_y),
    algorithm=ABBGKQLBM(lattice),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABBGKMeasurement(lattice),
    target_platform="QISKIT",
    compiler_platform="QISKIT",
    optimization_level=0,
    statevector_sampling=True,
    execution_backend=AerSimulator(method="statevector"),
    sampling_backend=AerSimulator(method="statevector"),
)

In [ ]:
cfg.prepare_for_simulation()

In [ ]:
# Number of shots to simulate for each timestep when running the circuit
NUM_SHOTS = 2**14

# Number of timesteps to simulate
NUM_STEPS = 10

Populations are recovered through the *square root* of a measured frequency, so shot
noise reaches the flow field amplified by $1/\sqrt{f_i}$. Passing
`save_statevector_to_disk=True` makes `ABBGKResult` post-process the exact amplitudes
instead, which is what the comparison below uses. The sampled fields remain available
through `ABBGKResult.counts_to_flow_field`.

In [ ]:
runner = QiskitRunner(cfg, lattice, save_statevector_to_disk=True)

result = runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)

## Diagnostics

The reinitializer reports how the state is distributed at the end of the last time step.
The physical sector holds most of the probability; the auxiliary sector holds the rest.
The two remaining entries must be zero: probability in an unused velocity label or on a
boundary-condition ancilla would mean a component acted on the wrong qubits.

In [ ]:
for name, value in runner.reinitializer.diagnostics.items():
    print(f"{name:<30} = {value:.3e}")

## Comparison with a classical solver

At $\tau=1$ the classical collision is a plain relaxation onto the equilibrium, which
makes a compact reference implementation possible. `D2Q9AngleEncoding` already provides
the equilibrium and the moments, so only streaming is left to write out.

In [ ]:
def classical_timestep(encoding, populations, solid_mask=None):
    """Advance one classical tau=1 D2Q9 BGK time step.

    At tau=1 the collision is simply a relaxation onto the local equilibrium, which is
    then streamed with periodic outer boundaries. Populations that would stream into a
    solid node are reflected back along the opposite channel, which is the same halfway
    bounce-back rule that ABReflectionOperator implements.
    """
    num_x, num_y, _ = populations.shape
    velocities = encoding.velocities.astype(int)
    opposite = [0, 3, 4, 1, 2, 7, 8, 5, 6]

    density, velocity_x, velocity_y = encoding.macroscopic(populations, solid_mask)
    collided = encoding.equilibrium(density, velocity_x, velocity_y)

    if solid_mask is not None:
        collided[solid_mask, :] = 0.0

    streamed = np.zeros_like(collided)

    for x in range(num_x):
        for y in range(num_y):
            if solid_mask is not None and solid_mask[x, y]:
                continue

            for velocity, (shift_x, shift_y) in enumerate(velocities):
                target_x, target_y = (x + shift_x) % num_x, (y + shift_y) % num_y

                if solid_mask is not None and solid_mask[target_x, target_y]:
                    streamed[x, y, opposite[velocity]] += collided[x, y, velocity]
                else:
                    streamed[target_x, target_y, velocity] += collided[x, y, velocity]

    return streamed

In [ ]:
populations = encoding.equilibrium(density, velocity_x, velocity_y)

for _ in range(NUM_STEPS):
    populations = classical_timestep(encoding, populations)

classical = encoding.macroscopic(populations)
quantum = (result.density, result.velocity_x, result.velocity_y)

velocity_error = np.linalg.norm(
    np.stack([quantum[1] - classical[1], quantum[2] - classical[2]])
) / np.linalg.norm(np.stack([classical[1], classical[2]]))

print(f"time steps                 = {NUM_STEPS}")
print(f"relative L2 velocity error = {velocity_error:.3e}")
print(
    f"relative L2 density error  = "
    f"{np.linalg.norm(quantum[0] - classical[0]) / np.linalg.norm(classical[0]):.3e}"
)

In [ ]:
def compare_flow_fields(quantum, classical):
    """Plot the quantum and classical speed fields side by side with their difference."""
    quantum_speed = np.sqrt(quantum[1] ** 2 + quantum[2] ** 2)
    classical_speed = np.sqrt(classical[1] ** 2 + classical[2] ** 2)
    error = np.sqrt((quantum[1] - classical[1]) ** 2 + (quantum[2] - classical[2]) ** 2)

    figure, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
    panels = (
        ("ABBGKQLBM", quantum_speed, "viridis"),
        ("Classical D2Q9 BGK", classical_speed, "viridis"),
        ("|u_quantum - u_classical|", error, "magma"),
    )
    speed_limit = max(quantum_speed.max(), classical_speed.max())

    for axis, (title, field, colormap) in zip(axes, panels):
        image = axis.imshow(
            field.T,
            origin="lower",
            cmap=colormap,
            vmin=0.0,
            vmax=speed_limit if colormap == "viridis" else None,
        )
        axis.set_title(title)
        axis.set_xlabel("x")
        axis.set_ylabel("y")
        figure.colorbar(image, ax=axis, shrink=0.85)

    return figure

In [ ]:
compare_flow_fields(quantum, classical);

## Decay of the vortex

Every time step is recorded in `ABBGKResult.flow_fields`, so the decay of the vortex can
be followed and checked against the analytical rate $\exp(-\nu k^2 t)$ with
$\nu = c_s^2 (\tau - 1/2) = 1/6$.

In [ ]:
steps = sorted(result.flow_fields)
peak_speed = [
    np.max(np.sqrt(result.flow_fields[step][1] ** 2 + result.flow_fields[step][2] ** 2))
    for step in steps
]

viscosity = (1.0 / 3.0) * (1.0 - 0.5)
analytical = INITIAL_SPEED * np.exp(-viscosity * 2 * wavenumber**2 * np.array(steps))

plt.figure(figsize=(6, 4))
plt.plot(steps, peak_speed, "o-", label="ABBGKQLBM")
plt.plot(steps, analytical, "k--", label=r"$U_0 e^{-2\nu k^2 t}$")
plt.xlabel("time step")
plt.ylabel("peak speed")
plt.title("Taylor-Green decay")
plt.legend()
plt.grid(alpha=0.3);

## Output

As with the other simulation demos, the ParaView files are written to the `paraview`
subdirectory of the output directory, one `step_<x>.vti` per time step. The scalar
stored in them is the velocity magnitude.

In [ ]:
print(f"ParaView output = {output_dir}/paraview")